# Function Calling Application
Function calling is a feature of the OpenAI Service designed to address the following challenges:
- Inconsistent Response Formatting: Before function calling, responses from a LLM were unstructured and inconsistent. Developers had to write complex validation code to handle each variation in the output.
- Limited Integration with External Data: Prior to this feature, it was difficult to incorporate data from other parts of an application into a chat context.

By standardizing response formats and enabling seamless integration with external data, function calling simplifies development and reduces the need for additional validation logic.


## 1. Understanding Function Calls

Users could not get answers like "What is the current weather in Stockholm?". This is because models were limited to the time the data was trained on. Let's look at the example below that illustrates this problem.

Let's say we want to create a database of student data so we can suggest the right course to them. Below we have two descriptions of students that are very similar in the data they contain.

In [40]:
# Define two students with data
student_1_description = f"""
Emily Johnson is a sophomore majoring in computer science at Duke University. 
She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team.
She hopes to pursue a career in software engineering after graduation.
"""
student_2_description = f"""
Michael Lee is a sophomore majoring in computer science at Stanford University. 
He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. 
He hopes to pursue a career in artificial intelligence after finishing his studies.
"""

We want to send this to an LLM to parse the data. This can later be used in our application to send this to an API or store it in a database.

Let's create two identical prompts that we instruct the LLM on what information that we are interested in. We want to send this to an LLM to parse the parts that are important to our product. 

In [41]:
# Create two identical prompts
prompt1 = f"""
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
"""

prompt2 = f"""
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
"""

After creating these two prompts, we will send them to the LLM by using `openai.chat.completion`. We store the prompt in the `messages` variable and assign the role to `user`. This is to mimic a message from a user being written to a chatbot.

In [42]:
# Import Libraries
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# Configure DeepSeek as used LLM
client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)
deployment="deepseek-v4-flash"

# Send both requests to the LLM and examine the response
openai_response1 = client.chat.completions.create(
    model=deployment,
    messages=[{"role": "user", "content": prompt1}]
)
response1 = openai_response1.choices[0].message.content

openai_response2 = client.chat.completions.create(
    model=deployment,
    messages=[{"role": "user", "content": prompt2}]
)
response2 = openai_response2.choices[0].message.content

# Load the response as a JSON object
json_response1 = json.loads(response1)
json_response2 = json.loads(response2)
print(json_response1)
print(json_response2)

{'name': 'Emily Johnson', 'major': 'computer science', 'school': 'Duke University', 'grades': '3.7 GPA', 'club': 'Chess Club and Debate Team'}
{'name': 'Michael Lee', 'major': 'computer science', 'school': 'Stanford University', 'grades': '3.8 GPA', 'club': 'Robotics Club'}


LLM takes the unstructured data in the form of the written prompt and returns also unstructured data. We need to have a structured format so that we know what to expect when storing or using this data.

By using functional calling, we can make sure that we receive structured data back. When using function calling, the LLM does not actually call or run any functions. Instead, we create a structure for the LLM to follow for its response. We then use those structured responses to know what function to run in our applications.

## 2. Creating your First Function Call

### Elements of a function call
THe first step is to create a user message. This can be dynamically assigned by taking the value of a text input or you can assign a value here. We need to define the `role` and the `content` of the message.

The `role` can be either `system` (creating rules), `assistant` (the model), or `user` (the end-user). For function calling, we will assign this as `user`.

### Creating Functions
Next we will define a function and the parameters of that function. We will use just one function here called `search_courses` but you can create multiple functions.

Important: Functions are included in the system message to the LLM and will be included in the amount of available tokens you have available.

In [43]:
function_messages = [{"role": "user", 
            "content": "Find me a good course for a beginner student to learn Azure."}]

functions = [
{
    "name": "search_courses",
    "description": "Retrieves courses from the search index based on the parameters provided",
    "parameters": {
        "type": "object",
        "properties" : {
            "role": {
                "type": "string",
                "description": "The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product": {
                "type": "string",
                "description": "The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level": {
                "type": "string",
                "description": "The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
        },
        "required": ["role"]
    }
}
]

tools = [
{
    "type": "function",
    "function": functions[0]
}
]

### Making the Function Call
After defining a function, we now need to include it in the call to the chat completion API. We do this by adding `functions` to the request. In this case `functions=functions`.

There is also an option to set `function_call` to `auto`. This means we will let the LLM decide which function should be called based on user message rather than assigning it ourselves.

In [44]:
# Get client response from LLm
first_response = client.chat.completions.create(
    model=deployment,
    messages=function_messages,
    tools=tools,
    tool_choice="auto"
)
#print(first_response.choices[0].message.content)
response_message = first_response.choices[0].message

print("\nThe response message from the first response is: ")
print(response_message)


The response message from the first response is: 
ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_8tsY29qDhheTKrk61fFI2107', function=Function(arguments='{"role": "student", "level": "beginner", "product": "Azure"}', name='search_courses'), type='function', index=0)], reasoning_content='The user is looking for a course for a beginner student to learn Azure. Let me search for courses with the appropriate parameters.')


## 3. Integrating Function Calls into an Application

After we have tested the formatted response from the LLM, now we can integrates this into an application.

In [45]:
import requests

# Define the function that will call the Microsoft Learn API to get a list of courses
def search_courses(role, product, level):
    print("Entered function call for Microsoft Learn!")
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    print("Microsoft Learn requests response: ")
    modules = response.json().get("modules", [])
    print(response.json())
    print(modules)
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)

As a best practice, we will then see if the model wants to call a function. After that, we will create one of the available functions and match it to the function that is being called. We will then take the arguments of the function and map them to arguments of from the LLM.

Lastly, we will append the function call message and the values that were returned by the `search_courses` message. This gives the LLM all the information it needs to respond to the user using natural language.

In [46]:
# Check if the model wants to call a function
if response_message.tool_calls:
    # Capture reasoning_content if present
    reasoning_content = getattr(response_message, "reasoning_content", None)

    # Check for tool_calls
    for tool_call in response_message.tool_calls:
        print("\nRecommended Function Call: ")
        print(tool_call.function.name)
        print()

        # Call the function
        function_name = tool_call.function.name
        available_functions = {
            "search_courses": search_courses,
        }
        function_to_call = available_functions[function_name]
        function_args = json.loads(tool_call.function.arguments)
        function_response = function_to_call(**function_args)

        # Print the function response
        print("\nOutput of Function Call: ")
        print(function_to_call)
        print(function_args)
        print(function_response)
        print(type(function_response))

        # Add the assistant response and function response to the messages
        function_messages.append(
        {
            "role": response_message.role,
            "reasoning_content": reasoning_content,
            "tool_calls": [
            {
                "id": tool_call.id,
                "type": "function",
                "function": {
                    "name": tool_call.function.name,
                    "arguments": tool_call.function.arguments
                }
            }
            ],
            "content": None,
        }
        )

        function_messages.append(
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": function_response,
        }
        )

# Print the messages
print("\nMessages in the second request: ")
print(function_messages)
print()

second_response = client.chat.completions.create(
    messages=function_messages,
    model=deployment,
    tools=tools,
    tool_choice="auto"
)
print(second_response.choices[0].message.content)
    


Recommended Function Call: 
search_courses

Entered function call for Microsoft Learn!
Microsoft Learn requests response: 
{'modules': [{'summary': 'Describe concepts of cryptography.', 'levels': ['beginner'], 'roles': ['business-owner', 'business-user', 'student'], 'products': ['azure', 'm365'], 'subjects': ['blockchain'], 'uid': 'learn.wwl.describe-concepts-of-cryptography', 'type': 'module', 'title': 'Describe concepts of cryptography', 'duration_in_minutes': 24, 'rating': {'count': 1955, 'average': 4.83}, 'popularity': 0.5881169254390797, 'icon_url': 'https://learn.microsoft.com/en-us/training/achievements/cryptography-describe-concepts.svg', 'social_image_url': 'https://learn.microsoft.com/en-us/training/achievements/generic-social.png', 'locale': 'en-us', 'last_modified': '2026-04-24T23:22:00+00:00', 'url': 'https://learn.microsoft.com/en-us/training/modules/describe-concepts-of-cryptography/?WT.mc_id=api_CatalogApi', 'firstUnitUrl': 'https://learn.microsoft.com/en-us/training/m